# Notebook 1: FAISS RAG Context Preparation
This notebook isolates the heavy 8B embedding model to pre-compute and retrieve the FAISS RAG contexts. This ensures zero VRAM overlap with the LLM generation phase.


## 1. Install Dependencies


In [ ]:
!pip install sentence-transformers faiss-cpu datasets scikit-learn


## 2. Load Dataset & Sample


In [ ]:
import json
import os
import random
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import faiss

test_file_path = "/kaggle/input/datasets/adityabayhaqie/nusantara-law-corpus/test-data-reformat.json"

test_data = []
if os.path.exists(test_file_path):
    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)
else:
    print(f"File not found: {test_file_path}")

# Extract unique context fields
unique_contexts = list(set([item.get('context', '').strip() for item in test_data if item.get('context', '').strip()]))
print(f"Found {len(unique_contexts)} unique contexts to build the FAISS index.")

# Seed and sample 20 evaluations
random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

eval_indices = random.sample(range(len(test_data)), min(20, len(test_data)))
eval_samples = [test_data[i] for i in eval_indices]
print(f"Selected {len(eval_samples)} samples for evaluation.")


## 3. Build FAISS Index with Qwen3-Embedding-8B


In [ ]:
print("Initializing Qwen/Qwen3-Embedding-8B in float16...")
embedder = SentenceTransformer(
    'Qwen/Qwen3-Embedding-8B', 
    model_kwargs={'torch_dtype': torch.float16}, 
    trust_remote_code=True
)

print("Encoding contexts to build FAISS index...")
context_embeddings = embedder.encode(unique_contexts, show_progress_bar=True)

dimension = context_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(context_embeddings)
print(f"FAISS index built successfully with {faiss_index.ntotal} contexts.")


## 4. Retrieve Contexts and Save to JSON


In [ ]:
def retrieve_contexts(query, k=5):
    query_emb = embedder.encode([query])
    D, I = faiss_index.search(query_emb, k=k)
    retrieved = [unique_contexts[idx] for idx in I[0] if idx != -1]
    return "\n\n".join(retrieved)

rag_eval_data = []
for sample in eval_samples:
    instruction = sample['instruction']
    retrieved_context = retrieve_contexts(instruction, k=5)
    rag_eval_data.append({
        'instruction': instruction,
        'original_context': sample.get('context', ''),
        'rag_context': retrieved_context,
        'response': sample['response']
    })

output_path = "/kaggle/working/rag_eval_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(rag_eval_data, f, indent=4, ensure_ascii=False)

print(f"Successfully pre-computed and saved {len(rag_eval_data)} RAG samples to {output_path}")
print("Notebook 1 execution complete! You can now run Notebook 2.")
